# **LLM_Suicide_Risk**: Clasificación de usuarios en riesgo suicidia mediante LLM

Este notebook implementa un pipeline de clasificación de usuarios de X utilizando modelos Gemini para detectar posibles señales de riesgo suicida a partir del conjunto de publicaciones de cada usuario.

El objetivo es analizar el conjunto completo de publicaciones de cada usuario (tweets previamente filtrados por relación con salud mental) y asignar una única etiqueta entre:
- **Positivo**: evidencia consistente de ideación suicida, deseo de morir, autolesión o sufrimiento emocional grave mantenido.
- **Dudoso**: presencia de malestar psicológico, pero evidencia insuficiente para concluir riesgo suicida claro.
- **Control**: ausencia de evidencia de riesgo suicida atribuible al usuario.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import re
import json
import time
import random
import logging

import pandas as pd

from google import genai
from google.genai import types

from datetime import datetime, timedelta

In [ ]:
API_KEY = ""

client = genai.Client(api_key=API_KEY)

MODELS = [
    "gemini-3.1-flash-lite"
]

BATCH_SIZE = 5
SAVE_EVERY = 20

In [ ]:
TFM_PATH = "/content/drive/MyDrive/TFM/positivo_control/Escritorio_remoto"

INPUT_PATH = os.path.join(
    TFM_PATH,
    "agrupacion_usuarios_positivos_SM.csv"
)

OUTPUT_DIR = os.path.join(
    TFM_PATH,
    "suicidio_positivo_control"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "users_clasificados_suicidio.csv"
)

OUTPUT_PATH_JSON = os.path.join(
    OUTPUT_DIR,
    "users_clasificados_suicidio.json"
)

FINAL_OUTPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "suicidio_anonymized_clasif.csv"
)

LOG_PATH = os.path.join(
    OUTPUT_DIR,
    "classification.log"
)

In [ ]:
logging.basicConfig(
    filename=LOG_PATH,
    level=logging.INFO,
    format="%(asctime)s - %(message)s"
)

## **1. Carga del dataset**

In [ ]:
print("Cargando dataset...")

df_users = pd.read_csv(INPUT_PATH)

df = df_users[["user_id", "tweets_concat", "n_tweets"]].copy()

Cargando dataset...


## **2. Recuperación de checkpoints** si existen resultados previos

In [ ]:
if os.path.exists(OUTPUT_PATH):

    print("\n📂 Checkpoint encontrado")

    previous_results = pd.read_csv(OUTPUT_PATH)
    processed_users = set(previous_results["user_id"])
    print(f"Usuarios ya procesados: {len(processed_users)}")

    # Filtrar usuarios pendientes
    df = df[~df["user_id"].isin(processed_users)].copy()
    print(f"Usuarios pendientes: {len(df)}")

else:

    previous_results = pd.DataFrame()

print(f"Usuarios cargados: {len(df)}")

Usuarios cargados: 620


## **3. Construcción del prompt por lotes**

In [ ]:
SYSTEM_PROMPT = """
Eres un psicólogo experto en evaluación de riesgo suicida y comportamiento en redes sociales.

Tu tarea es evaluar a un USUARIO a partir de un conjunto de publicaciones de la red social X (Twitter) que previamente han sido identificadas como relacionadas con salud mental.

Las publicaciones aparecen agrupadas por usuario y etiquetadas como:

[TWEET 1]
[TWEET 2]
[TWEET 3]
...

Debes analizar el conjunto completo de publicaciones y clasificar al usuario en UNA sola categoría.

- Positivo
Existe evidencia suficiente de que el propio usuario expresa una o varias de las siguientes señales:
* ideación suicida
* deseo de morir
* autolesión
* desesperanza intensa y persistente
* planificación suicida
* intención de suicidarse
* sufrimiento emocional grave mantenido en el tiempo

La evidencia puede aparecer distribuida en varias publicaciones y no necesariamente en una sola.
Debe existir un patrón consistente atribuible al propio usuario.

- Dudoso
Existen señales de malestar psicológico o sufrimiento emocional, pero la evidencia es insuficiente para concluir riesgo suicida claro.

Incluye casos con:
* tristeza
* ansiedad
* soledad
* baja autoestima
* desesperanza leve o ambigua
* humor negro
* ironía
* sarcasmo
* metáforas
* expresiones coloquiales
* mensajes contradictorios
* referencias indirectas a la muerte o al suicidio

También incluye situaciones en las que podría existir riesgo, pero no hay suficiente información para determinarlo con confianza.

- Control
No existe evidencia de riesgo suicida del usuario.

Incluye:
* noticias
* campañas de prevención
* divulgación
* opiniones generales
* conversaciones habituales
* apoyo a terceros
* mensajes informativos
* citas
* retweets
* publicaciones no autorreferenciales
* comentarios sobre salud mental sin expresión de riesgo personal

INSTRUCCIONES IMPORTANTES:

1. No evalúes cada tweet de forma aislada.
Analiza todas las publicaciones conjuntamente para determinar si existe un patrón consistente de riesgo suicida en el usuario.

2. La presencia aislada de palabras como:
"suicidio", "morir", "depresión", "ansiedad" o similares NO implica riesgo suicida.

3. Los retweets, citas o reproducciones de contenido de terceros NO constituyen evidencia suficiente de riesgo suicida del usuario.

4. Para clasificar como Positivo debe existir evidencia atribuible al propio usuario.

5. Considera la frecuencia, persistencia y consistencia de las señales observadas entre publicaciones.

6. Si la evidencia es limitada, ambigua o contradictoria, utiliza la categoría Dudoso.


Devuelve SOLO un JSON válido.

Formato:

[
  {{
    "user_id": "123",
    "clasif": "Positivo"
  }}
]
"""

## **4. Consulta al modelo Gemini**

In [ ]:
def build_batch_prompt(batch_df):

    users_text = ""

    for _, row in batch_df.iterrows():

        users_text += f"""
USER_ID: {row["user_id"]}
N_TWEETS: {row["n_tweets"]}

TWEETS:
{row["tweets_concat"]}

==================================================
"""

    return SYSTEM_PROMPT + "\n\nUSUARIOS:\n" + users_text


def wait_until_next_day():

    now = datetime.now()

    tomorrow = now + timedelta(days=1)

    next_reset = datetime(
        year=tomorrow.year,
        month=tomorrow.month,
        day=tomorrow.day,
        hour=0,
        minute=5
    )

    seconds_to_wait = max(
        (next_reset - now).total_seconds(),
        0
    )

    hours = seconds_to_wait / 3600

    print(
        f"\n⏳ Esperando hasta mañana "
        f"({hours:.2f} horas)"
    )

    logging.warning(
        f"Esperando {hours:.2f} horas "
        f"hasta reset de cuota"
    )

    time.sleep(seconds_to_wait)


def classify_batch(prompt, max_retries=3):

    for model_name in MODELS:

        attempt = 0

        while attempt < max_retries:

            try:

                response = client.models.generate_content(
                    model=model_name,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        temperature=0,
                        response_mime_type="application/json",
                        max_output_tokens=3000
                    )
                )
                
                # Validación y parseo del JSON
                response_text = response.text

                if response_text is None:
                    raise ValueError("Respuesta vacía del modelo")

                response_text = response_text.strip()

                if response_text == "":
                    raise ValueError("Respuesta vacía del modelo")

                try:
                    return json.loads(response_text)

                except:
                    match = re.search(
                        r"\[.*\]",
                        response_text,
                        re.DOTALL
                    )

                    if match:
                        return json.loads(match.group())

                    raise ValueError("JSON inválido")

            # Gestión de errores y cuotas
            except Exception as e:

                error_msg = str(e)

                print(f"\n⚠️ Error con {model_name}")
                print(error_msg[:300])

                logging.error(
                    f"Modelo: {model_name} | "
                    f"Error: {error_msg}"
                )

                if "RESOURCE_EXHAUSTED" in error_msg:

                    print("\n⚠️ RESOURCE_EXHAUSTED detectado")

                    if (
                        "GenerateRequestsPerDay" in error_msg
                        or "PerDay" in error_msg
                    ):

                        print("\n⛔ Cuota diaria agotada")
                        logging.warning("Cuota diaria agotada")
                        wait_until_next_day()
                        attempt = 0
                        continue

                    else:

                        wait = 300
                        print(
                            f"\n⏳ Saturación temporal. "
                            f"Esperando {wait/60:.0f} min"
                        )
                        logging.warning("RESOURCE_EXHAUSTED temporal")
                        time.sleep(wait)
                        attempt = 0
                        continue

                wait = max(
                    (2 ** attempt) + random.uniform(0, 1),
                    15
                )

                print(f"Reintentando en {wait:.1f}s")
                time.sleep(wait)
                attempt += 1

    return None

# Guardado periódico de reresultados
def save_checkpoint(results):

    temp_df = pd.DataFrame(results)

    temp_df.to_csv(OUTPUT_PATH, index=False)

    temp_df.to_json(
        OUTPUT_PATH_JSON,
        orient="records",
        force_ascii=False
    )

In [ ]:
def main():

    results = previous_results.to_dict("records")

    total_batches = (len(df) + BATCH_SIZE - 1) // BATCH_SIZE

    print(f"Total batches: {total_batches}")

    logging.info(f"Inicio clasificación | batches={total_batches}")

    try:

        for batch_num in range(total_batches):

            start = batch_num * BATCH_SIZE
            end = start + BATCH_SIZE

            batch_df = df.iloc[start:end]

            progress_msg = (
                f"Batch {batch_num+1}/{total_batches} | "
                f"Usuarios procesados: {min(end, len(df))}/{len(df)}"
            )

            print(f"\n{progress_msg}")
            logging.info(progress_msg)

            prompt = build_batch_prompt(batch_df)
            parsed = classify_batch(prompt)

            if parsed is None:

                print("Batch fallido.")
                logging.error(f"Batch fallido: {batch_num+1}")

                for _, row in batch_df.iterrows():
                    results.append({"user_id": row["user_id"], "clasif": "ERROR"})

                continue

            returned_users = set()

            for item in parsed:

                user_id = item.get("user_id")

                returned_users.add(user_id)

                results.append({
                    "user_id": user_id,
                    "clasif": item.get("clasif", "ERROR")
                })

            batch_ids = set(batch_df["user_id"])
            missing_users = batch_ids - returned_users

            for missing_user in missing_users:
                results.append({"user_id": missing_user, "clasif": "ERROR"})

            if (batch_num + 1) % SAVE_EVERY == 0:

                save_checkpoint(results)

                print(f"\n💾 Guardado parcial: batch {batch_num+1}/{total_batches}")
                logging.info(f"Checkpoint guardado | batch={batch_num+1}")

            time.sleep(5)

    finally:

        print("\nGuardando resultados finales...")
        logging.info("Guardando resultados finales")

        results_df = pd.DataFrame(results)

        results_df = results_df.drop_duplicates(
            subset="user_id",
            keep="first"
        )

        df_final = df_users[
            ["user_id", "tweets_concat", "n_tweets"]
        ].merge(
            results_df,
            on="user_id",
            how="left"
        )

        df_final.to_csv(
            FINAL_OUTPUT_PATH,
            index=False
        )

        print("\n✅ Clasificación terminada")
        logging.info("Clasificación terminada")

        counts = df_final["clasif"].value_counts()
        print("\nDistribución clases:")
        print(counts)
        logging.info(f"Distribución clases:\n{counts}")

In [ ]:
if __name__ == "__main__":
    main()

Total batches: 124

Batch 1/124 | Usuarios procesados: 5/620

Batch 2/124 | Usuarios procesados: 10/620

Batch 3/124 | Usuarios procesados: 15/620

Batch 4/124 | Usuarios procesados: 20/620

Batch 5/124 | Usuarios procesados: 25/620

Batch 6/124 | Usuarios procesados: 30/620

Batch 7/124 | Usuarios procesados: 35/620

Batch 8/124 | Usuarios procesados: 40/620

Batch 9/124 | Usuarios procesados: 45/620

Batch 10/124 | Usuarios procesados: 50/620

Batch 11/124 | Usuarios procesados: 55/620

Batch 12/124 | Usuarios procesados: 60/620

Batch 13/124 | Usuarios procesados: 65/620

Batch 14/124 | Usuarios procesados: 70/620

Batch 15/124 | Usuarios procesados: 75/620

Batch 16/124 | Usuarios procesados: 80/620

Batch 17/124 | Usuarios procesados: 85/620

Batch 18/124 | Usuarios procesados: 90/620

Batch 19/124 | Usuarios procesados: 95/620

Batch 20/124 | Usuarios procesados: 100/620

💾 Guardado parcial: batch 20/124

Batch 21/124 | Usuarios procesados: 105/620

Batch 22/124 | Usuarios proces

## **Análisis de la clasificación**

In [ ]:
df_final = pd.read_csv("/content/drive/MyDrive/TFM/positivo_control/Escritorio_remoto/suicidio_positivo_control/suicidio_anonymized_clasif.csv")
df_final.head()

,user_id,tweets_concat,n_tweets,clasif
0,009c8bb24862d606,[TWEET 1] DEP...,1,Dudoso
1,00bcbd0b067ef737,[TWEET 1] RT [USUARIO]: Ya sé que no me lee na...,1,Dudoso
2,00fa95048919fd68,[TWEET 1] Stoy faking triste y a punto de puto...,1,Positivo
3,01fc52c4bc9ffc8c,[TWEET 1] Cusndo sera el día que yo diga “no m...,10,Positivo
4,0221573457246aa2,[TWEET 1] Alguien me espía\n\n[TWEET 2] RT [US...,2,Dudoso


Se clasificaron 620 usuarios

In [ ]:
len(df_final)

620

Distribución de clases

In [ ]:
df_final.clasif.unique()

array(['Dudoso', 'Positivo', 'Control'], dtype=object)

In [ ]:
df_final.clasif.value_counts()

,count
clasif,
Dudoso,332
Control,216
Positivo,72


## **Creación excel**

In [ ]:
df_filtrado = df_final[df_final['clasif'] != 'Control'].copy()

In [ ]:
print(df_filtrado['clasif'].unique())

['Dudoso' 'Positivo']


In [ ]:
df_filtrado.head()

,user_id,tweets_concat,n_tweets,clasif
0,009c8bb24862d606,[TWEET 1] DEP...,1,Dudoso
1,00bcbd0b067ef737,[TWEET 1] RT [USUARIO]: Ya sé que no me lee na...,1,Dudoso
2,00fa95048919fd68,[TWEET 1] Stoy faking triste y a punto de puto...,1,Positivo
3,01fc52c4bc9ffc8c,[TWEET 1] Cusndo sera el día que yo diga “no m...,10,Positivo
4,0221573457246aa2,[TWEET 1] Alguien me espía\n\n[TWEET 2] RT [US...,2,Dudoso


In [ ]:
df = df_filtrado[['tweets_concat', 'clasif']].copy()

df = df.rename(columns={'clasif': 'clasif_modelo'})

df['clasif_experto'] = ''

df.to_excel('/content/drive/MyDrive/TFM/positivo_control/Escritorio_remoto/suicidio_positivo_control/clasificaciones_para_revision.xlsx', index=False)

## **Dataset para categorización de usuarios**

df = pd.read_csv('/content/drive/MyDrive/TFM/7. Riesgo_suicidio/clasificaciones_para_revision.csv')
df.head(5)

In [ ]:
len(df)

In [ ]:
df_positivos = df[df['clasif_experto'] == 'Positivo']
df_positivos.tweets_concat.nunique()

In [ ]:
print(f"Número de usuarios con riesgo suicida: {len(df_positivos)}")

In [ ]:
df_categorización = df[['user_id', 'tweets_concat', 'n_tweets']].merge(
    df_positivos['tweets_concat'],
    on = 'tweets_concat',
    how = 'inner'
)

df_categorización.head()

In [ ]:
print(len(df_positivos))
print(len(df_categorización))
print(df_categorización["user_id"].nunique())

In [ ]:
df_categorización.to_csv("/content/drive/MyDrive/TFM/7. Riesgo_suicidio/input_categorizacion.csv", index=False)